In [ ]:
# Cleaning Riders Dataset

import pandas as pd
import numpy as np

# Load raw riders dataset
riders = pd.read_csv(
    "../data/raw/riders.csv",
    parse_dates=["signup_date"]
)

In [2]:
# Create a copy before cleaning
riders_clean = riders.copy()

In [3]:
# Remove duplicate rows
riders_clean = riders_clean.drop_duplicates()

In [4]:

# Standardise text columns
text_cols = ["loyalty_status", "city", "referred_by"]

for col in text_cols:
    riders_clean[col] = (
        riders_clean[col]
        .astype("string")
        .str.strip()
    )

In [5]:

# Fill missing referral values
riders_clean["referred_by"] = riders_clean["referred_by"].fillna("unknown")

In [6]:
# Create referral flag
riders_clean["is_referred"] = (
    riders_clean["referred_by"] != "unknown"
).astype(int)


In [8]:
# Validate age range
riders_clean = riders_clean[
    riders_clean["age"].between(18, 100)
]

In [9]:

# Validate rating range
riders_clean = riders_clean[
    riders_clean["avg_rating_given"].between(0, 5)
]

In [10]:

# Validate churn probability range
riders_clean = riders_clean[
    riders_clean["churn_prob"].between(0, 1)
]

In [11]:
# Create binary churn target variable
riders_clean["is_churned"] = (
    riders_clean["churn_prob"] >= 0.5
).astype(int)

In [12]:
# Check final cleaned dataset
print("Original shape:", riders.shape)
print("Cleaned shape:", riders_clean.shape)
print("Duplicate rows:", riders_clean.duplicated().sum())
print("Missing values:")
print(riders_clean.isnull().sum())

Original shape: (10000, 8)
Cleaned shape: (10000, 10)
Duplicate rows: 0
Missing values:
user_id             0
signup_date         0
loyalty_status      0
age                 0
city                0
avg_rating_given    0
churn_prob          0
referred_by         0
is_referred         0
is_churned          0
dtype: int64


In [13]:
# Display the cleaned dataset
riders_clean.head()

,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,referred_by,is_referred,is_churned
0,R00000,2025-01-24,Bronze,34.729629,Nairobi,5.0,0.142431,R00001,1,0
1,R00001,2024-09-09,Bronze,34.571020,Nairobi,4.7,0.674161,unknown,0,1
2,R00002,2024-09-07,Bronze,47.133960,Lagos,4.2,0.510379,unknown,0,1
3,R00003,2025-03-17,Bronze,41.658628,Nairobi,4.9,0.244779,unknown,0,0
4,R00004,2024-08-20,Silver,40.681709,Lagos,3.9,0.269960,R00002,1,0


In [14]:

# Save cleaned riders dataset
riders_clean.to_csv(
    "../data/processed/riders_clean.csv",
    index=False
)

print("Riders dataset cleaned and saved successfully.")

Riders dataset cleaned and saved successfully.


    Cleaning the trips

In [18]:
import pandas as pd
import numpy as np

data_files = {
    "sessions": ["session_time"],
    "trips": ["pickup_time", "dropoff_time"],
    "riders": ["signup_date"],
    "drivers": ["signup_date", "last_active"],
    "promotions": ["start_date", "end_date"]
}

dfs = {}

for name, date_cols in data_files.items():
    dfs[name] = pd.read_csv(
        f"../data/raw/{name}.csv",
        parse_dates=date_cols
    )

sessions = dfs["sessions"]
trips = dfs["trips"]
riders = dfs["riders"]
drivers = dfs["drivers"]
promotions = dfs["promotions"]

In [19]:
# Create a copy of the trips dataset
trips_clean = dfs["trips"].copy()

In [20]:
# Convert pickup and dropoff times to UTC datetime
trips_clean["pickup_time"] = pd.to_datetime(
    trips_clean["pickup_time"],
    format="mixed",
    utc=True
)

trips_clean["dropoff_time"] = pd.to_datetime(
    trips_clean["dropoff_time"],
    format="mixed",
    utc=True
)

In [21]:
# Keep only trips where dropoff time is after pickup time
trips_clean = trips_clean[
    trips_clean["dropoff_time"] > trips_clean["pickup_time"]
]

In [22]:
# Create trip duration in minutes
trips_clean["trip_duration_minutes"] = (
    trips_clean["dropoff_time"] - trips_clean["pickup_time"]
).dt.total_seconds() / 60

In [23]:
# Check trip duration summary
trips_clean["trip_duration_minutes"].describe()

count    200000.000000
mean         31.957310
std          15.871754
min           5.000000
25%          18.000000
50%          32.000000
75%          46.000000
max          59.000000
Name: trip_duration_minutes, dtype: float64

In [24]:
# Remove duplicate rows
trips_clean = trips_clean.drop_duplicates()

In [25]:
# Remove duplicate trip IDs
trips_clean = trips_clean.drop_duplicates(subset="trip_id")

In [26]:
# Keep only trips where dropoff time is after pickup time
trips_clean = trips_clean[
    trips_clean["dropoff_time"] > trips_clean["pickup_time"]
]

In [27]:
# Keep trips with non-negative fares
trips_clean = trips_clean[
    trips_clean["fare"] >= 0
]

In [28]:
# Keep trips with non-negative tips
trips_clean = trips_clean[
    trips_clean["tip"] >= 0
]

In [29]:
# Keep valid surge multipliers
trips_clean = trips_clean[
    trips_clean["surge_multiplier"] >= 1
]

In [30]:
# Calculate trip duration in minutes
trips_clean["trip_duration_minutes"] = (
    trips_clean["dropoff_time"] -
    trips_clean["pickup_time"]
).dt.total_seconds() / 60

In [31]:
# Keep realistic trip durations
trips_clean = trips_clean[
    trips_clean["trip_duration_minutes"].between(1, 300)
]

In [32]:
# Display original and cleaned dataset shapes
print("Original shape:", dfs["trips"].shape)
print("Cleaned shape:", trips_clean.shape)

print()

# Check for duplicate rows
print("Duplicate rows:", trips_clean.duplicated().sum())

# Check for duplicate Trip IDs
print("Duplicate Trip IDs:", trips_clean["trip_id"].duplicated().sum())

print()

# Display missing values
print("Missing values:")
print(trips_clean.isnull().sum())

Original shape: (200000, 16)
Cleaned shape: (200000, 17)

Duplicate rows: 0
Duplicate Trip IDs: 0

Missing values:
trip_id                  0
user_id                  0
driver_id                0
fare                     0
surge_multiplier         0
tip                      0
payment_type             0
pickup_time              0
dropoff_time             0
pickup_lat               0
pickup_lng               0
dropoff_lat              0
dropoff_lng              0
weather                  0
city                     0
loyalty_status           0
trip_duration_minutes    0
dtype: int64


In [52]:
# ==================================================
# Convert Pickup and Dropoff Time to Datetime
# ==================================================

trips["pickup_time"] = pd.to_datetime(
    trips["pickup_time"],
    utc=True
)

trips["dropoff_time"] = pd.to_datetime(
    trips["dropoff_time"],
    utc=True
)

# ==================================================
# Create Trip Duration (Minutes)
# ==================================================

trips["trip_duration_minutes"] = (
    trips["dropoff_time"] - trips["pickup_time"]
).dt.total_seconds() / 60

In [53]:
# ==================================================
# Validate Trip Duration
# ==================================================

print(trips["trip_duration_minutes"].describe())

print(
    "Negative durations:",
    (trips["trip_duration_minutes"] < 0).sum()
)

count    200000.000000
mean         31.957310
std          15.871754
min           5.000000
25%          18.000000
50%          32.000000
75%          46.000000
max          59.000000
Name: trip_duration_minutes, dtype: float64
Negative durations: 0


In [54]:
# Remove trips with negative duration
trips = trips[
    trips["trip_duration_minutes"] >= 0
]

In [55]:
# Save the cleaned trips dataset
trips_clean.to_csv(
    "../data/processed/trips_clean.csv",
    index=False
)

print("Trips dataset cleaned and saved successfully.")

Trips dataset cleaned and saved successfully.


In [56]:
# Display the first five rows of the cleaned trips dataset
trips_clean.head()

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration_minutes
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50+00:00,2024-11-27 17:06:50+00:00,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48+00:00,2024-10-28 23:12:48+00:00,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41+00:00,2025-02-17 03:25:41+00:00,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14+00:00,2024-06-18 17:27:14+00:00,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16+00:00,2024-10-05 08:01:16+00:00,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0


In [57]:
# Save cleaned trips dataset
trips_clean.to_csv("../data/processed/trips_clean.csv", index=False)

print("Trips dataset cleaned and saved successfully.")

Trips dataset cleaned and saved successfully.


Cleaning the 'Driver'

In [58]:
# Create a copy of the original drivers dataset
drivers_clean = drivers.copy()

# Store original shape before cleaning
original_shape = drivers_clean.shape

# Remove exact duplicate rows
drivers_clean = drivers_clean.drop_duplicates()

# Remove duplicate driver IDs, keeping the first record
drivers_clean = drivers_clean.drop_duplicates(subset=["driver_id"], keep="first")

# Convert date columns to datetime format
drivers_clean["signup_date"] = pd.to_datetime(drivers_clean["signup_date"], errors="coerce")
drivers_clean["last_active"] = pd.to_datetime(drivers_clean["last_active"], errors="coerce")

# Remove records with missing important fields
drivers_clean = drivers_clean.dropna(subset=[
    "driver_id",
    "rating",
    "vehicle_type",
    "signup_date",
    "last_active",
    "city",
    "acceptance_rate"
])

# Validate rating range
drivers_clean = drivers_clean[
    drivers_clean["rating"].between(1, 5)
]

# Validate acceptance rate range
drivers_clean = drivers_clean[
    drivers_clean["acceptance_rate"].between(0, 1)
]

# Ensure last_active is not before signup_date
drivers_clean = drivers_clean[
    drivers_clean["last_active"] >= drivers_clean["signup_date"]
]

# Standardise text columns
drivers_clean["vehicle_type"] = drivers_clean["vehicle_type"].str.strip().str.title()
drivers_clean["city"] = drivers_clean["city"].str.strip().str.title()

# Final summary
print("Original shape:", original_shape)
print("Cleaned shape:", drivers_clean.shape)

print("\nDuplicate rows:", drivers_clean.duplicated().sum())
print("Duplicate Driver IDs:", drivers_clean["driver_id"].duplicated().sum())

drivers["signup_date"] = pd.to_datetime(drivers["signup_date"])

drivers["last_active"] = pd.to_datetime(drivers["last_active"])

print("\nMissing values:")
print(drivers_clean.isnull().sum())

display(drivers_clean.head())

Original shape: (5000, 7)
Cleaned shape: (4772, 7)

Duplicate rows: 0
Duplicate Driver IDs: 0

Missing values:
driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773
5,D00005,3.1,Sedan,2024-09-27,2024-12-15 23:26:07.576316,Nairobi,0.874726


In [59]:
# Save cleaned drivers dataset
drivers_clean.to_csv("../data/processed/drivers_clean.csv", index=False)

print("Drivers dataset cleaned and saved successfully.")

Drivers dataset cleaned and saved successfully.


Clean Sessions Dataset

In [ ]:
# Clean Sessions Dataset

# Create a copy
sessions_clean = sessions.copy()

# Remove duplicate rows
sessions_clean = sessions_clean.drop_duplicates()

# Standardise text columns
text_cols = ["city", "loyalty_status"]

for col in text_cols:
    sessions_clean[col] = (
        sessions_clean[col]
        .astype(str)
        .str.strip()
        .str.title()
    )

# Convert session time to datetime (if needed)
sessions_clean["session_time"] = pd.to_datetime(
    sessions_clean["session_time"],
    errors="coerce",
    utc=True
)

# Remove invalid time_on_app values
sessions_clean = sessions_clean[
    sessions_clean["time_on_app"] >= 0
]

# Validate pages visited
sessions_clean = sessions_clean[
    sessions_clean["pages_visited"].between(1, 5)
]

# Validate converted values
sessions_clean = sessions_clean[
    sessions_clean["converted"].isin([0, 1])
]

In [39]:
sessions_clean.sort_values(
    by="time_on_app",
    ascending=False
).head(10)

,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
24342,S024342,R06544,2025-04-27 06:50:18+00:00,1800,3,0,Nairobi,Gold
20090,S020090,R03001,2025-04-27 20:09:46+00:00,1800,4,0,Nairobi,Bronze
1069,S001069,R01281,2025-04-27 08:45:31+00:00,1800,2,0,Cairo,Silver
24303,S024303,R07699,2025-04-27 10:43:08+00:00,1800,5,0,Nairobi,Silver
41644,S041644,R03009,2025-04-27 13:24:23+00:00,1800,5,0,Nairobi,Bronze
8484,S008484,R07629,2025-04-27 05:02:29+00:00,1800,1,0,Nairobi,Bronze
15968,S015968,R01193,2025-04-27 14:09:20+00:00,1800,2,0,Cairo,Bronze
11712,S011712,R03800,2025-04-27 07:24:58+00:00,1800,3,1,Cairo,Bronze
5685,S005685,R01816,2025-04-27 01:47:15+00:00,1800,3,0,Nairobi,Platinum
5676,S005676,R05930,2025-04-27 04:51:58+00:00,1800,1,0,Lagos,Silver


In [40]:
sessions_clean.to_csv(
    "../data/processed/sessions_clean.csv",
    index=False
)

Cleaning the promotions dataset

In [41]:
# Create a copy of the promotions dataset
promotions_clean = dfs["promotions"].copy()

In [42]:
# Convert start_date and end_date to datetime
promotions_clean["start_date"] = pd.to_datetime(promotions_clean["start_date"])
promotions_clean["end_date"] = pd.to_datetime(promotions_clean["end_date"])

In [43]:
# Remove duplicate rows
promotions_clean = promotions_clean.drop_duplicates()

In [44]:
# Remove duplicate promotion IDs
promotions_clean = promotions_clean.drop_duplicates(subset="promo_id")

In [45]:
# Keep only promotions where end_date is after start_date
promotions_clean = promotions_clean[
    promotions_clean["end_date"] > promotions_clean["start_date"]
]

In [46]:
# Keep only valid promotion values
promotions_clean = promotions_clean[
    promotions_clean["promo_value"] > 0
]

In [47]:
# Create promotion duration in days
promotions_clean["promo_duration_days"] = (
    promotions_clean["end_date"] - promotions_clean["start_date"]
).dt.days

In [48]:
# Standardise text columns by removing extra spaces
text_cols = ["promo_name", "promo_type", "target_segment"]

for col in text_cols:
    promotions_clean[col] = promotions_clean[col].str.strip()

In [49]:
# Validate cleaned promotions dataset
print("Original shape:", dfs["promotions"].shape)
print("Cleaned shape:", promotions_clean.shape)

print()

print("Duplicate rows:", promotions_clean.duplicated().sum())
print("Duplicate promo IDs:", promotions_clean["promo_id"].duplicated().sum())

print()

print("Missing values:")
print(promotions_clean.isnull().sum())

Original shape: (20, 11)
Cleaned shape: (20, 12)

Duplicate rows: 0
Duplicate promo IDs: 0

Missing values:
promo_id               0
promo_name             0
promo_type             0
promo_value            0
start_date             0
end_date               0
target_segment         0
city_scope             0
ab_test_groups         0
test_allocation        0
success_metric         0
promo_duration_days    0
dtype: int64


In [50]:
# Display cleaned promotions dataset
promotions_clean.head()

,promo_id,promo_name,promo_type,promo_value,start_date,end_date,target_segment,city_scope,ab_test_groups,test_allocation,success_metric,promo_duration_days
0,P000,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-25,All,Nairobi,['All'],[1.0],Usage Frequency,29
1,P001,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-22,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate,26
2,P002,Peak Hour Pass,surge_waiver,1.0,2025-04-26,2025-05-16,All,Cairo,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",ROI,20
3,P003,Loyalty Bonus,points,100.0,2025-04-26,2025-05-04,Gold+,Nairobi,"['Control', 'Variant A', 'Variant B']","[0.3, 0.4, 0.3]",Conversion Rate,8
4,P004,Loyalty Bonus,points,100.0,2025-04-26,2025-05-15,Gold+,Nairobi,['All'],[1.0],Usage Frequency,19


In [51]:
# Save cleaned promotions dataset
promotions_clean.to_csv(
    "../data/processed/promotions_clean.csv",
    index=False
)

print("Promotions dataset cleaned and saved successfully.")

Promotions dataset cleaned and saved successfully.
